# 🐉 SOV33 · Hunyuan3D-2.1 → real Hatch character meshes (Colab T4)

Generate **sovereign-owned 3D character meshes** (`.glb`) for SovSpace from the reference art or a
text prompt — the one genuinely-new capability that was GPU-gated. Feeds the EXISTING pipeline:
`meok-os-deploy/character.html` already loads `.glb`/VRM via three.js `GLTFLoader` — this fills the
gap where characters were RPM/VRM-loaded externally with **no sovereign-generated mesh**.

**Honest:** Hunyuan3D-2.1 is open-weights but heavy. On a free **T4 (16 GB)** use the **mini/turbo**
shape variant; the full model wants A100. Output is a textured `.glb` you download, sign via
`os.meok.ai/api/provenance`, and serve statically → loaded free in the WebGL body (client-GPU).

Pipeline: **image (ref art or text→image) → Hunyuan3D shape (DiT) → texture → `.glb` → sign → serve.**

In [ ]:
# 1. GPU check — need a T4/L4/A100. Runtime > Change runtime type > T4 GPU.
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout or 'NO GPU — set Runtime>T4')

In [ ]:
# 2. Install Hunyuan3D-2.1 (Tencent, open-weights). ~5-8 min on first run.
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1 || echo 'exists'
%cd Hunyuan3D-2.1
!pip install -q -r requirements.txt 2>/dev/null; pip install -q trimesh rembg onnxruntime huggingface_hub
# compile the texture/rasterizer custom ops (needed for textured output)
!cd hy3dpaint/custom_rasterizer && pip install -q -e . 2>/dev/null || echo 'rasterizer skip (shape-only still works)'
print('✅ Hunyuan3D-2.1 installed')

In [ ]:
# 3. Provide the source image: upload the sovereign reference art (meok-3d-characters/ref_dragon_hatchling.png)
#    OR generate one from a text prompt first (uncomment the text2img block).
from google.colab import files
print('Upload ref_dragon_hatchling.png (or any single character image):')
up = files.upload()
IMG = list(up.keys())[0]

# --- optional: text -> image (if you have no ref art) ---
# from diffusers import AutoPipelineForText2Image; import torch
# t2i = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16).to('cuda')
# img = t2i('a sovereign dragon hatchling, clean studio background, full body, centered', num_inference_steps=4, guidance_scale=0).images[0]
# IMG='hatch_gen.png'; img.save(IMG)
print('source image:', IMG)

In [ ]:
# 4. SHAPE: image -> 3D mesh (DiT). Uses the mini/turbo variant so it fits a T4.
import torch
from PIL import Image
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from hy3dshape.rembg import BackgroundRemover

img = Image.open(IMG).convert('RGBA')
img = BackgroundRemover()(img)   # isolate the character

pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1', subfolder='hunyuan3d-dit-v2-1', variant='fp16')
mesh = pipe(image=img, num_inference_steps=30, octree_resolution=256,
            guidance_scale=5.0, generator=torch.manual_seed(1))[0]
mesh.export('hatch_shape.glb')
print('✅ shape mesh -> hatch_shape.glb  (', len(mesh.vertices), 'verts )')

In [ ]:
# 5. TEXTURE: paint the mesh from the source image -> final textured .glb
try:
    from hy3dpaint.textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
    paint = Hunyuan3DPaintPipeline(Hunyuan3DPaintConfig(max_num_view=6, resolution=512))
    out = paint(mesh_path='hatch_shape.glb', image_path=IMG, output_mesh_path='hatch_textured.glb')
    FINAL='hatch_textured.glb'
    print('✅ textured mesh -> hatch_textured.glb')
except Exception as e:
    FINAL='hatch_shape.glb'
    print('texture step skipped (', str(e)[:80], ') — shipping shape-only glb:', FINAL)

In [ ]:
# 6. SIGN + download. Sovereign provenance on the generated asset, then serve it.
import requests, json, hashlib, os
sha = hashlib.sha256(open(FINAL,'rb').read()).hexdigest()
try:
    prov = requests.get('https://os.meok.ai/api/provenance', params={
        'claim': f'MEOK Hatch character mesh {FINAL} sha256={sha[:16]}',
        'source': 'Hunyuan3D-2.1 on Colab T4', 'kind': 'generated-3d-asset'}, timeout=20).json()
    json.dump(prov, open('hatch_mesh.provenance.json','w'), indent=2)
    print('✅ signed provenance:', prov.get('signature','')[:24], '…')
except Exception as e:
    print('provenance skipped:', str(e)[:60])
print('sha256:', sha)
from google.colab import files
files.download(FINAL)
# → commit to meok-os-deploy/models/hatch.glb and point character.html's GLTFLoader at /models/hatch.glb
print('\nNEXT: drop', FINAL, '-> meok-os-deploy/models/hatch.glb ; character.html already loads .glb via GLTFLoader.')